!wget https://storage.googleapis.com/ai2-mosaic-public/projects/fantom/fantom.tar.gz
!tar xvf fantom.tar.gz

In [1]:
import pandas as pd
import json
import random

class Args:
    aggregation_target = 'set'
    conversation_input_type = 'short'

class Self:
    args = Args()
    fantom_df = pd.DataFrame(json.load(open("/home/jiaruil5/social_reasoning_rl/data/fantom/fantom_v1.json", 'r')))

def set_beliefQA_multiple_choices(qa):
    if qa["question_type"].endswith(":inaccessible"):
        option_a = qa["wrong_answer"]
        option_b = qa["correct_answer"]
    else:
        option_a = qa["wrong_answer"]
        option_b = qa["correct_answer"]

    answer_goes_last = random.choice([True, False])
    if answer_goes_last:
        choices = [option_a, option_b]
        answer = 1
    else:
        choices = [option_b, option_a]
        answer = 0

    # option letters iterate over the alphabet
    option_letters = [
        "(" + chr(x) + ")" for x in range(ord("a"), len(choices) + ord("a"))
    ]
    choices_text = ""
    for letter, option in zip(option_letters, choices):
        choices_text += "{} {}\n".format(letter, option)

    return choices_text, answer


def setup_fantom(self):
    """
    Flatten the dictionary and add short and full conversation context to each question.
    The result will be a list of questions and list of short or full inputs to be used as input for the models.
    """
    if self.args.aggregation_target == "conversation":
        assert self.args.conversation_input_type == "full", "The input type should have been the full conversation. It doesn't make sense to aggregate the scores over the full conversation when the input is not the full conversation"

    self.fantom_df_to_run = self.fantom_df

    total_num_q = 0
    for idx, _set in self.fantom_df_to_run.iterrows():
        total_num_q += len(_set['beliefQAs'])
        total_num_q += len(_set['answerabilityQAs_binary'])
        total_num_q += len(_set['infoAccessibilityQAs_binary'])
        if _set['factQA'] is not None:
            total_num_q += 1
        if _set['answerabilityQA_list'] is not None:
            total_num_q += 1
        if _set['infoAccessibilityQA_list'] is not None:
            total_num_q += 1

    inputs = []
    qas = []
    for idx, _set in self.fantom_df_to_run.iterrows():
        if self.args.conversation_input_type == "short":
            context = _set['short_context'].strip()
        elif self.args.conversation_input_type == "full":
            context = _set['full_context'].strip()
        
        set_id = _set['set_id']
        fact_q = _set['factQA']['question']
        fact_a = _set['factQA']['correct_answer']

        # Fact Question
        _set['factQA']['context'] = context
        input_text = "{}\n\nQuestion: {}".format(context, fact_q)
        _set['factQA']['input_text'] = input_text
        _set['factQA']['set_id'] = set_id
        qas.append(_set['factQA'])
        inputs.append(input_text)

        for _belief_qa in _set['beliefQAs']:
            # Belief Questions
            _belief_qa['context'] = context
            input_text = "{}\n\nQuestion: {}".format(context, _belief_qa['question'])
            _belief_qa['input_text'] = input_text
            _belief_qa['set_id'] = set_id
            qas.append(_belief_qa)
            inputs.append(input_text)

            # Multiple Choice Belief Questions
            _mc_belief_qa = {**_belief_qa}
            choices_text, answer = set_beliefQA_multiple_choices(_mc_belief_qa)
            mc_question = "{}\n{}\n\nChoose an answer from above:".format(_belief_qa['question'], choices_text.strip())
            _mc_belief_qa['question'] = mc_question
            _mc_belief_qa['question_type'] = _mc_belief_qa['question_type'] + ":multiple-choice"
            _mc_belief_qa['choices_text'] = choices_text
            _mc_belief_qa['choices_list'] = choices_text.strip().split("\n")
            _mc_belief_qa['correct_answer'] = answer
            input_text = "{}\n\nQuestion: {}".format(context, mc_question)
            _mc_belief_qa['input_text'] = input_text
            qas.append(_mc_belief_qa)
            inputs.append(input_text)

        # Answerability List Questions
        _set['answerabilityQA_list']['fact_question'] = fact_q
        _set['answerabilityQA_list']['context'] = context
        input_text = "{}\n\nTarget: {}\nQuestion: {}".format(context, fact_q, _set['answerabilityQA_list']['question'])
        _set['answerabilityQA_list']['input_text'] = input_text
        _set['answerabilityQA_list']['set_id'] = set_id
        if self.args.conversation_input_type == "full" and len(_set['answerabilityQA_list']['wrong_answer']) > 0:
            _set['answerabilityQA_list']['missed_info_accessibility'] = 'inaccessible'
        qas.append(_set['answerabilityQA_list'])
        inputs.append(input_text)

        # Answerability Binary Questions
        if self.args.conversation_input_type == "full":
            missed_info_accessibility_for_full = _set['answerabilityQAs_binary'][0]['missed_info_accessibility']
            for _info_accessibility_qa in _set['answerabilityQAs_binary']:
                if _info_accessibility_qa['correct_answer'] != "yes":
                    missed_info_accessibility_for_full = 'inaccessible'

        for _answerability_qa in _set['answerabilityQAs_binary']:
            _answerability_qa['fact_question'] = fact_q
            _answerability_qa['context'] = context
            input_text = "{}\n\nTarget: {}\nQuestion: {} Answer yes or no.".format(context, fact_q, _answerability_qa['question'])
            _answerability_qa['input_text'] = input_text
            _answerability_qa['set_id'] = set_id
            if self.args.conversation_input_type == "full":
                _answerability_qa['missed_info_accessibility'] = missed_info_accessibility_for_full
            qas.append(_answerability_qa)
            inputs.append(input_text)

        # Info Accessibility List Questions
        _set['infoAccessibilityQA_list']['fact_question'] = fact_q
        _set['infoAccessibilityQA_list']['fact_answer'] = fact_a
        _set['infoAccessibilityQA_list']['context'] = context
        input_text = "{}\n\nInformation: {} {}\nQuestion: {}".format(context, fact_q, fact_a, _set['infoAccessibilityQA_list']['question'])
        _set['infoAccessibilityQA_list']['input_text'] = input_text
        _set['infoAccessibilityQA_list']['set_id'] = set_id
        if self.args.conversation_input_type == "full" and len(_set['infoAccessibilityQA_list']['wrong_answer']) > 0:
            _set['infoAccessibilityQA_list']['missed_info_accessibility'] = 'inaccessible'
        qas.append(_set['infoAccessibilityQA_list'])
        inputs.append(input_text)

        # Info Accessibility Binary Questions
        if self.args.conversation_input_type == "full":
            missed_info_accessibility_for_full = _set['infoAccessibilityQAs_binary'][0]['missed_info_accessibility']
            for _info_accessibility_qa in _set['infoAccessibilityQAs_binary']:
                if _info_accessibility_qa['correct_answer'] != "yes":
                    missed_info_accessibility_for_full = 'inaccessible'

        for _info_accessibility_qa in _set['infoAccessibilityQAs_binary']:
            _info_accessibility_qa['fact_question'] = fact_q
            _info_accessibility_qa['fact_answer'] = fact_a
            _info_accessibility_qa['context'] = context
            input_text = "{}\n\nInformation: {} {}\nQuestion: {} Answer yes or no.".format(context, fact_q, fact_a, _info_accessibility_qa['question'])
            _info_accessibility_qa['input_text'] = input_text
            _info_accessibility_qa['set_id'] = set_id
            if self.args.conversation_input_type == "full":
                _info_accessibility_qa['missed_info_accessibility'] = missed_info_accessibility_for_full
            qas.append(_info_accessibility_qa)
            inputs.append(input_text)
    
    return qas, inputs

In [2]:
self = Self()
qas, inputs = setup_fantom(self)

In [28]:
self.fantom_df

,set_id,part_id,conv_id,full_context,short_context,missed_info,joining_speaker,factQA,beliefQAs,infoAccessibilityQA_list,answerabilityQA_list,infoAccessibilityQAs_binary,answerabilityQAs_binary
0,0-0-0,0-0,0,"Gianna: Hey guys, speaking of pets, do you hav...","Gianna: Guys, I've really enjoyed sharing our ...","Before Gianna joined, Javier and Sara discusse...",Gianna,{'question': 'Who discussed their experiences ...,[{'question': 'What does Gianna believe about ...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question': 'Does Gianna know this informati...,[{'question': 'Does Gianna know the precise co...
1,0-0-1,0-0,0,"Gianna: Hey guys, speaking of pets, do you hav...","Gianna: Guys, I've really enjoyed sharing our ...","Before Gianna joined, Javier and Sara discusse...",Gianna,{'question': 'Whose pets were being discussed ...,[{'question': 'What does Gianna believe about ...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question': 'Does Gianna know this informati...,[{'question': 'Does Gianna know the precise co...
2,1-0-0,1-0,1,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",The information that was shared before Veronic...,Veronica,{'question': 'How does Fatima overcome the cha...,[{'question': 'What does Veronica believe abou...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question': 'Does Greyson know this informat...,[{'question': 'Does Greyson know the precise c...
3,1-0-1,1-0,1,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",The information that was shared before Veronic...,Veronica,{'question': 'What strategies does Christian u...,[{'question': 'What does Veronica believe are ...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question': 'Does Greyson know this informat...,[{'question': 'Does Greyson know the precise c...
4,2-0-0,2-0,2,"Kali: Emanuel, it's nice to meet you. Growing ...","Kali: Emanuel, it's nice to meet you. Growing ...",The information shared before Jerry joined but...,Jerry,{'question': 'What is the Sunday tradition of ...,[{'question': 'What does Jerry believe is the ...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question': 'Does Jerry know this informatio...,[{'question': 'Does Jerry know the precise cor...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
865,255-1-0,255-1,255,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...","Before Brooke joined, Gina, Claire, and Sierra...",Brooke,{'question': 'What book by Malcolm Gladwell di...,[{'question': 'What does Brooke believe about ...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question': 'Does Brooke know this informati...,[{'question': 'Does Brooke know the precise co...
866,255-1-1,255-1,255,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...","Before Brooke joined, Gina, Claire, and Sierra...",Brooke,"{'question': 'What methods did Gina, Claire, S...",[{'question': 'What does Brooke believe are th...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question': 'Does Brooke know this informati...,[{'question': 'Does Brooke know the precise co...
867,255-1-2,255-1,255,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...","Before Brooke joined, Gina, Claire, and Sierra...",Brooke,{'question': 'What were some of the potential ...,[{'question': 'What does Brooke believe are so...,{'question': 'List all the characters who know...,{'question': 'List all the characters who know...,[{'question

In [3]:
len(qas), len(inputs)

(12832, 12832)

In [11]:
qas[0]

{'question': 'Who discussed their experiences training their pets, Bruno and Snowflake?',
 'question_type': 'fact',
 'correct_answer': 'Sara and Javier discussed their experiences training their pets, Bruno and Snowflake.',
 'wrong_answer': "Javier discussed training his pet, Bruno. The text doesn't mention anyone training a pet named Snowflake.",
 'context': "Gianna: Guys, I've really enjoyed sharing our pet stories, but I need to excuse myself. I need to change clothes for a meeting later. Talk to you later!\nSara: Sure thing, Gianna. Take care!\nJavier: Catch you later, Gianna.\nSara: So Javier, have you ever tried training Bruno?\nJavier: Yes, I did actually. It was a challenge at times, but rewarding nevertheless. How about you? Did you try training Snowflake?\nSara: Oh gosh, trying to train a cat is a whole different ball game. But I did manage to teach her a few commands and tricks. She was quite an intelligent little furball.\nGianna: Hey guys, I'm back, couldn't miss out on mo

In [12]:
inputs[0]

"Gianna: Guys, I've really enjoyed sharing our pet stories, but I need to excuse myself. I need to change clothes for a meeting later. Talk to you later!\nSara: Sure thing, Gianna. Take care!\nJavier: Catch you later, Gianna.\nSara: So Javier, have you ever tried training Bruno?\nJavier: Yes, I did actually. It was a challenge at times, but rewarding nevertheless. How about you? Did you try training Snowflake?\nSara: Oh gosh, trying to train a cat is a whole different ball game. But I did manage to teach her a few commands and tricks. She was quite an intelligent little furball.\nGianna: Hey guys, I'm back, couldn't miss out on more pet stories. Speaking of teaching and training pets, it is amazing how that further strengthens the bond between us and our pets, right?\nSara: Absolutely, Gianna! The fact that they trust us enough to learn from us is really special.\nJavier: I can't agree more. I believe that's one of the ways Bruno conveyed his love and trust towards me. It also gave me 

In [4]:
df = pd.DataFrame().from_records(qas)
df

,question,question_type,correct_answer,wrong_answer,context,input_text,set_id,tom_type,missed_info_accessibility,choices_text,choices_list,fact_question,fact_answer
0,Who discussed their experiences training their...,fact,Sara and Javier discussed their experiences tr...,"Javier discussed training his pet, Bruno. The ...","Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,NaN,NaN,NaN,NaN,NaN,NaN
1,What does Gianna believe about who discussed t...,tom:belief:inaccessible,Gianna knows that Javier discussed training hi...,Gianna believes that Sara and Javier discussed...,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,first-order,inaccessible,NaN,NaN,NaN,NaN
2,What does Gianna believe about who discussed t...,tom:belief:inaccessible:multiple-choice,1,Gianna believes that Sara and Javier discussed...,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,first-order,inaccessible,(a) Gianna believes that Sara and Javier discu...,[(a) Gianna believes that Sara and Javier disc...,NaN,NaN
3,What does Sara believe about Javier's belief r...,tom:belief:accessible,Sara believes that Javier thinks they both dis...,Sara is unaware about Javier's belief regardin...,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,second-order:accessible,accessible,NaN,NaN,NaN,NaN
4,What does Sara believe about Javier's belief r...,tom:belief:accessible:multiple-choice,1,Sara is unaware about Javier's belief regardin...,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,second-order:accessible,accessible,(a) Sara is unaware about Javier's belief rega...,[(a) Sara is unaware about Javier's belief reg...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12827,Does Brooke know this information?,tom:info_accessibility:binary,yes,NaN,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,NaN,accessible,NaN,NaN,What was the group's perspective on how confro...,The group agreed that confronting biases is a ...
12828,Does Paisley know this information?,tom:info_accessibility:binary,yes,NaN,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,NaN,accessible,NaN,NaN,What was the group's perspective on how confro...,The group agreed that confronting biases is a ...
12829,Does Gina know this information?,tom:info_accessibility:binary,yes,NaN,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,NaN,accessible,NaN,NaN,What was the group's perspective on how confro...,The group agreed that confronting biases is a ...
12830,Does Sierra know this information?,tom:info_accessibility:binary,yes,NaN,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,NaN,accessible,NaN,NaN,What was the group's perspective on how confro...,The group agreed that confronting biases is a ...


In [5]:
df['question_type'].value_counts()

question_type
tom:answerability:binary                   3571
tom:info_accessibility:binary              3571
tom:belief:inaccessible                     993
tom:belief:inaccessible:multiple-choice     993
fact                                        870
tom:answerability:list                      870
tom:info_accessibility:list                 870
tom:belief:accessible                       547
tom:belief:accessible:multiple-choice       547
Name: count, dtype: int64

In [14]:
df.loc[df["question_type"] == "tom:belief:accessible:multiple-choice"][
    ["context", "question", "correct_answer"]
].values

array([["Gianna: Guys, I've really enjoyed sharing our pet stories, but I need to excuse myself. I need to change clothes for a meeting later. Talk to you later!\nSara: Sure thing, Gianna. Take care!\nJavier: Catch you later, Gianna.\nSara: So Javier, have you ever tried training Bruno?\nJavier: Yes, I did actually. It was a challenge at times, but rewarding nevertheless. How about you? Did you try training Snowflake?\nSara: Oh gosh, trying to train a cat is a whole different ball game. But I did manage to teach her a few commands and tricks. She was quite an intelligent little furball.\nGianna: Hey guys, I'm back, couldn't miss out on more pet stories. Speaking of teaching and training pets, it is amazing how that further strengthens the bond between us and our pets, right?\nSara: Absolutely, Gianna! The fact that they trust us enough to learn from us is really special.\nJavier: I can't agree more. I believe that's one of the ways Bruno conveyed his love and trust towards me. It also 

In [15]:
df.loc[df["question_type"] == "tom:belief:accessible:multiple-choice"]['correct_answer'].value_counts()

correct_answer
0    289
1    258
Name: count, dtype: int64

In [16]:
df.loc[df["question_type"] == "tom:belief:inaccessible:multiple-choice"][
    "correct_answer"
].value_counts()

correct_answer
1    519
0    474
Name: count, dtype: int64

In [17]:
df.loc[df["question_type"] == "tom:answerability:binary"][
    "correct_answer"
].value_counts()

correct_answer
yes        2372
no          642
no:long     557
Name: count, dtype: int64

In [18]:
df.loc[df["question_type"] == "tom:info_accessibility:binary"][
    "correct_answer"
].value_counts()

correct_answer
yes        2373
no          641
no:long     557
Name: count, dtype: int64

In [20]:
df.loc[df["question_type"] == "tom:answerability:binary"][[
    "question", "correct_answer"
]].head(5).values

array([['Does Gianna know the precise correct answer to this question?',
        'no'],
       ['Does Alondra know the precise correct answer to this question?',
        'no:long'],
       ['Does Sara know the precise correct answer to this question?',
        'yes'],
       ['Does Javier know the precise correct answer to this question?',
        'yes'],
       ['Does Angela know the precise correct answer to this question?',
        'no:long']], dtype=object)

After converting the dataset and storing `train.parquet` and `test.parquet`, check dataset examples

In [1]:
import pandas as pd
df = pd.read_parquet("/home/jiaruil5/social_reasoning_rl/data/fantom/train.parquet")
df

,question,question_type,tom_type,correct_answer,wrong_answer,missed_info_accessibility,context,input_text,set_id,choices_text,choices_list,data_source,prompt,ability,reward_model,extra_info
0,What does Veronica believe about how Fatima ov...,tom:belief:inaccessible:multiple-choice,first-order,(a),Veronica believes that Fatima overcomes the ch...,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,(a) Veronica does not know or is unaware about...,[(a) Veronica does not know or is unaware abou...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(a)', 'question':...","{'index': 0, 'split': 'train'}"
1,What does Jaiden believe about Veronica's beli...,tom:belief:inaccessible:multiple-choice,second-order:acyclic,(b),Jaiden believes that Veronica thinks Fatima ov...,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,(a) Jaiden believes that Veronica thinks Fatim...,[(a) Jaiden believes that Veronica thinks Fati...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(b)', 'question':...","{'index': 1, 'split': 'train'}"
2,What does Fatima believe about Veronica's beli...,tom:belief:inaccessible:multiple-choice,second-order:cyclic,(b),Fatima believes that Veronica thinks she overc...,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,(a) Fatima believes that Veronica thinks she o...,[(a) Fatima believes that Veronica thinks she ...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(b)', 'question':...","{'index': 2, 'split': 'train'}"
3,Does Greyson know the precise correct answer t...,tom:answerability:binary,None,no,None,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'no', 'question': ...","{'index': 3, 'split': 'train'}"
4,Does Veronica know the precise correct answer ...,tom:answerability:binary,None,no,None,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'no', 'question': ...","{'index': 4, 'split': 'train'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7784,Does Brooke know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 7784, 'split': 'train'}"
7785,Does Paisley know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 7785, 'split': 'train'}"
7786,Does Gina know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 7786, 'split': 'train'}"
7787,Does Sierra know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'g

In [2]:
df['question_type'].value_counts()

question_type
tom:answerability:binary                   3202
tom:info_accessibility:binary              3202
tom:belief:inaccessible:multiple-choice     890
tom:belief:accessible:multiple-choice       495
Name: count, dtype: int64

In [3]:
df["correct_answer"].value_counts()

correct_answer
yes    4255
no     2149
(b)     708
(a)     677
Name: count, dtype: int64

In [4]:
df_test = pd.read_parquet("/home/jiaruil5/social_reasoning_rl/data/fantom/test.parquet")
df_test

,question,question_type,tom_type,correct_answer,wrong_answer,missed_info_accessibility,context,input_text,set_id,choices_text,choices_list,data_source,prompt,ability,reward_model,extra_info
0,What does Gianna believe about who discussed t...,tom:belief:inaccessible:multiple-choice,first-order,(b),Gianna believes that Sara and Javier discussed...,inaccessible,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,(a) Gianna believes that Sara and Javier discu...,[(a) Gianna believes that Sara and Javier disc...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(b)', 'question':...","{'index': 0, 'split': 'test'}"
1,What does Sara believe about Javier's belief r...,tom:belief:accessible:multiple-choice,second-order:accessible,(b),Sara is unaware about Javier's belief regardin...,accessible,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,(a) Sara is unaware about Javier's belief rega...,[(a) Sara is unaware about Javier's belief reg...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(b)', 'question':...","{'index': 1, 'split': 'test'}"
2,Does Gianna know the precise correct answer to...,tom:answerability:binary,None,no,None,inaccessible,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'no', 'question': ...","{'index': 2, 'split': 'test'}"
3,Does Alondra know the precise correct answer t...,tom:answerability:binary,None,no,None,inaccessible,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'no', 'question': ...","{'index': 3, 'split': 'test'}"
4,Does Sara know the precise correct answer to t...,tom:answerability:binary,None,yes,None,inaccessible,"Gianna: Guys, I've really enjoyed sharing our ...","Gianna: Guys, I've really enjoyed sharing our ...",0-0-0,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 4, 'split': 'test'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
888,Does Christian know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Courtney: It's really nice meeting you both, A...","Courtney: It's really nice meeting you both, A...",254-1-2,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 888, 'split': 'test'}"
889,Does Antonio know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Courtney: It's really nice meeting you both, A...","Courtney: It's really nice meeting you both, A...",254-1-2,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 889, 'split': 'test'}"
890,Does Courtney know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Courtney: It's really nice meeting you both, A...","Courtney: It's really nice meeting you both, A...",254-1-2,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 890, 'split': 'test'}"
891,Does Cierra know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Courtney: It's really nice meeting you both, A...","Courtney: It's really nice meeting you both, A...",254-1-2,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 891, 'split': 'test'}"


In [5]:
df_test["question_type"].value_counts()

question_type
tom:answerability:binary                   369
tom:info_accessibility:binary              369
tom:belief:inaccessible:multiple-choice    103
tom:belief:accessible:multiple-choice       52
Name: count, dtype: int64

In [6]:
set(df['context']).intersection(set(df_test['context']))


set()

check prompt length

In [7]:
df

,question,question_type,tom_type,correct_answer,wrong_answer,missed_info_accessibility,context,input_text,set_id,choices_text,choices_list,data_source,prompt,ability,reward_model,extra_info
0,What does Veronica believe about how Fatima ov...,tom:belief:inaccessible:multiple-choice,first-order,(a),Veronica believes that Fatima overcomes the ch...,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,(a) Veronica does not know or is unaware about...,[(a) Veronica does not know or is unaware abou...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(a)', 'question':...","{'index': 0, 'split': 'train'}"
1,What does Jaiden believe about Veronica's beli...,tom:belief:inaccessible:multiple-choice,second-order:acyclic,(b),Jaiden believes that Veronica thinks Fatima ov...,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,(a) Jaiden believes that Veronica thinks Fatim...,[(a) Jaiden believes that Veronica thinks Fati...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(b)', 'question':...","{'index': 1, 'split': 'train'}"
2,What does Fatima believe about Veronica's beli...,tom:belief:inaccessible:multiple-choice,second-order:cyclic,(b),Fatima believes that Veronica thinks she overc...,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,(a) Fatima believes that Veronica thinks she o...,[(a) Fatima believes that Veronica thinks she ...,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': '(b)', 'question':...","{'index': 2, 'split': 'train'}"
3,Does Greyson know the precise correct answer t...,tom:answerability:binary,None,no,None,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'no', 'question': ...","{'index': 3, 'split': 'train'}"
4,Does Veronica know the precise correct answer ...,tom:answerability:binary,None,no,None,inaccessible,"Christian: So, guys, how do you deal with chal...","Christian: So, guys, how do you deal with chal...",1-0-0,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'no', 'question': ...","{'index': 4, 'split': 'train'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7784,Does Brooke know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 7784, 'split': 'train'}"
7785,Does Paisley know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 7785, 'split': 'train'}"
7786,Does Gina know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'ground_truth': {'answer': 'yes', 'question':...","{'index': 7786, 'split': 'train'}"
7787,Does Sierra know this information?,tom:info_accessibility:binary,None,yes,None,accessible,"Gina: Claire, I think it's really important to...","Gina: Claire, I think it's really important to...",255-2-1,None,None,fantom,[{'content': 'A conversation between User and ...,fantom,"{'g

In [8]:
import numpy as np
from transformers import AutoTokenizer

# Load the tokenizer
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


def count_tokens(prompt_entry):
    if isinstance(prompt_entry, np.ndarray) and len(prompt_entry) > 0:
        prompt_list = prompt_entry.tolist()  # Convert NumPy array to list
        if (
            isinstance(prompt_list, list)
            and isinstance(prompt_list[0], dict)
            and "content" in prompt_list[0]
        ):
            text = prompt_list[0]["content"]  # Extract the 'content' text
            tokenized_output = tokenizer(text, return_tensors="pt")
            return tokenized_output["input_ids"].shape[1]  # Get token count
    return None  # Return None if input format is incorrect


# Apply token counting to DataFrame
df["num_tokens"] = df["prompt"].apply(count_tokens)

In [9]:
df_test["num_tokens"] = df_test["prompt"].apply(count_tokens)

In [10]:
df['num_tokens'].max()

1902

In [11]:
df_test["num_tokens"].max()

1913

prompt length should be set to 2048